# E10: simulated usage sessions, transcribed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e10_sessions_colab.ipynb)

Experiment E10 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
analyses usage logs the way a product team would. Real logs are not available, so the sessions are
**simulated** from the FLEURS dev sets, keeping apart what is simulated and what is real:

- **simulated**: which sessions exist, i.e. the mix of languages, channels (headset or phone) and
  environments (quiet, stationary noise, babble), drawn with a fixed seed from
  [`experiments/E10/simulation.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E10/simulation.toml);
- **real**: the recordings, the noise and the telephone channel applied to them, and the errors
  and confidence of the model that transcribes them.

The 3,000 sessions are listed in the committed manifest `experiments/E10/sessions.jsonl`. This
notebook rebuilds their audio, checks that the plan is the committed one, and transcribes them
with Whisper-large-v3-turbo and the decoding options selected in E2. The analysis runs locally
from the saved predictions.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes
   about 45 minutes; keep the tab open.
2. At the end the browser downloads `e10_run.zip` (if it does not, download it from the *Files*
   panel). It contains `predictions.jsonl` and `run.json`, no audio.
3. In a local clone, from the repository root: `python3 -m zipfile -e e10_run.zip .`

If the session is interrupted, run all cells again: audio already written is kept and the
transcription resumes where it stopped.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(Path.cwd())

## 2. Settings

In [ ]:
SIMULATION_PATH = Path("experiments/E10/simulation.toml")  # the simulated traffic mix
SESSIONS_PATH = Path("experiments/E10/sessions.jsonl")  # the committed plan of the sessions
SWEEP_PATH = Path("experiments/E2/sweep.toml")  # the model and its decoding options...
SELECTION_PATH = Path("experiments/E2/results/selection_dev.json")  # ...as selected in E2
RUN_DIR = Path("experiments/runs/e10-turbo-sessions")

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The FLEURS dev sets, built exactly as on a local machine. The sessions are drawn again from the
spec and must be identical to the committed plan; then their audio is written (a few minutes).

In [ ]:
from asr_lab.analytics.sessions import load_simulation_spec, plan_sessions, render_sessions
from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.manifest import read_manifest

simulation = load_simulation_spec(SIMULATION_PATH)
for lang in simulation.languages:
    manifest = simulation.source_manifest(lang)
    if not manifest.exists():
        ingest_fleurs(lang, "dev", manifest.parent)
if plan_sessions(simulation) != read_manifest(SESSIONS_PATH):
    raise RuntimeError(f"{SESSIONS_PATH} is not the plan of {SIMULATION_PATH}")
print(f"{render_sessions(SESSIONS_PATH)} session(s) rendered")

## 4. Transcription

All sessions with one model, the E2 selection (about 35 minutes).

In [ ]:
import json

from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.run import transcribe_manifest
from asr_lab.sweep import load_spec

sweep = load_spec(SWEEP_PATH)
selected = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))["selected"]
options = sweep.config(selected).options
print(f"{sweep.model_size}, E2 selection {selected!r}: {options}")
model = FasterWhisperModel(sweep.model_size, device="cuda", **options)
transcribe_manifest(model, SESSIONS_PATH, RUN_DIR)

## 5. Download the predictions

In [ ]:
import zipfile

run = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
print(f"{run['n_utterances']} sessions, RTF {run['rtf'] or float('nan'):.3f}")
archive = Path("e10_run.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for name in ("predictions.jsonl", "run.json"):
        zipped.write(RUN_DIR / name, (RUN_DIR / name).as_posix())
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass